In [5]:
# Import libraries
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt 
import joblib

In [6]:
# Load dataset

X_train = np.load('../data/processed/X_train.npy')
X_test = np.load('../data/processed/X_test.npy')

input_dim = X_train.shape[1]
print(f'Input features: {input_dim}')

Input features: 21


In [7]:
from tensorflow.keras import layers, models, regularizers

input_dim = X_train.shape[1] 

print("Building Autoencoder with L1 Regularization...")

# Define the architecture
input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(16, activation='relu')(input_layer)

# L1 Regularizer (1e-5) is applied ONLY to the bottleneck
encoded = layers.Dense(8, activation='relu', activity_regularizer=regularizers.l1(1e-5))(encoded)

decoded = layers.Dense(16, activation='relu')(encoded)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

Building Autoencoder with L1 Regularization...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 21)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 21)             │           357 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 989 (3.86 KB)

 Trainable params: 989 (3.86 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Starting training...")
history= autoencoder.fit(
    X_train, X_train,   #input is X_train, target is also X_train
    epochs = 50,
    batch_size = 256,
    validation_split = 0.2,
    callbacks = [early_stopping],
    verbose = 1
)

# saving the trained model
autoencoder.save("../models/autoencoder.keras")
print("Model Saved")


Starting training...
Epoch 1/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 383us/step - loss: 0.0062 - val_loss: 0.0017
Epoch 2/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 351us/step - loss: 0.0013 - val_loss: 0.0011
Epoch 3/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 350us/step - loss: 9.6830e-04 - val_loss: 8.8816e-04
Epoch 4/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 352us/step - loss: 8.5306e-04 - val_loss: 8.9831e-04
Epoch 5/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 355us/step - loss: 7.9656e-04 - val_loss: 7.4239e-04
Epoch 6/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 352us/step - loss: 7.5993e-04 - val_loss: 7.2227e-04
Epoch 7/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 359us/step - loss: 7.3192e-04 - val_loss: 7.1113e-04
Epoch 8/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 355us/step - loss: 7.1372e-04 - val_loss: 6.7276e-04
Epoch 9/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 352us/step - loss: 6.9579e-04 - val_loss: 6.6237e-04
Epoch 10/50
5683/5683 ━━━━━━━━━━━━━━━━━━━━ 2s 358us/step - loss: 6.8157e-04 - val_loss: 6.5049e-04
Epoch 11/50
56